
<a id="summary"></a>

## Notebook 3 — RFM scoring & clustering (behavioral segments)

**v1.** This notebook engineers **RFM features** at **customer** grain (on top of completed transactions), prepares **clustering** on behavioral patterns, and targets roadmap **Q6–Q10** from `docs/fintech-ai-segmentation-summary.md`.

### How this notebook is organized

**Jump to parts:** [Part 1 — RFM design & data loading](#part-1) · Part 2 — RFM metrics & scores *(later)* · Part 3 — clustering & segment profiles *(later)* · Part 4 — products & utilization *(later)*

**Part 1 — RFM design & data loading**

1. [RFM ranking criteria (locked definitions)](#rfm-criteria)
2. [Load raw tables from Supabase](#q1)
3. [Join transactions to customer attributes](#q2)
4. [Transaction month buckets & incomplete last month](#q3)

---

### Objectives

- **This pass:** Document **RFM criteria**, import libraries, load **`customers_raw`**, **`transactions_raw`**, **`products_raw`**, **`customer_products_raw`**, build **`df_tx`** with the same **incomplete trailing month** rule as notebook 2, and pin **`as_of_date`** for downstream 12-month windows.
- **Later:** Quintile scores, K-Means / segments, answers to Q6–Q10 (including product counts and credit utilization when features exist).

### Business questions (Behavioral Intelligence)

6. Who are our most valuable customers right now?
7. What behavioral patterns define each customer segment?
8. What is the RFM profile of each segment?
9. Do high-engagement customers own more products?
10. How does credit utilization vary across segments? *(requires engineered `customers_features` or a documented proxy — see [RFM criteria](#rfm-criteria).)*

### Expected output

- **`df_tx`**: completed transactions joined to customers, **after** dropping an incomplete latest calendar month when applicable.
- **`as_of_date`**: reference instant for **recency** and the **12-month** frequency/monetary window.
- **`df_products`**, **`df_customer_products`**: for product-ownership analysis (Q9).
- Later: per-customer **`recency_days`**, **`frequency_12m`**, **`monetary_12m`**, quintile scores, **`rfm_score`**, **`predicted_segment`**.



<a id="rfm-criteria"></a>

### RFM ranking criteria (v1)

These definitions stay aligned with [`2.EDA_cohort_analysis.ipynb`](2.EDA_cohort_analysis.ipynb) calendar rules and with [`docs/notebooks-roadmap.md`](docs/notebooks-roadmap.md) Notebook 4.

| Topic | Rule |
|-------|------|
| **Calendar completeness** | Before computing RFM, drop the **latest calendar month** in `transaction_datetime` if coverage is incomplete: require at least **`min(20, days_in_month)`** distinct calendar days with at least one **completed** transaction in that month. Recompute `transaction_month` / `registration_month` after the drop. |
| **`as_of_date`** | **`max(transaction_datetime)`** on the retained **`df_tx`** rows (after the filter above). All RFM windows are relative to this snapshot. |
| **Recency (R)** | Days from **`as_of_date`** to each customer's **last** completed transaction timestamp (`>= 0`). Customers with no transaction in the retained data are out of scope for RFM until handled explicitly. |
| **Frequency (F)** | Count of completed transactions with `transaction_datetime` in **(as_of_date − 12 months, as_of_date]** (12-month lookback, inclusive of the last instant). |
| **Monetary (M)** | Sum of **`amount`** over those same rows. Exclude **`transaction_type = 'refund'`** from F and M when that column is present. |
| **Scores 1–5** | **Quintiles** on each dimension among eligible customers: higher score = more recent (lower days), higher frequency, higher monetary. Ties: use `method='first'` in `pandas.qcut` unless we document otherwise. |
| **Combined `rfm_score`** | For ordering: **`100 * R_score + 10 * F_score + M_score`** (so lexicographic R → F → M). For reporting, also keep the three scores separate. |
| **Q10 — Credit utilization** | Not present on `customers_raw`. The unified view in [`supabase/customers_analytics_view.sql`](../../supabase/customers_analytics_view.sql) expects **`customers_features.credit_utilization_pct`**. Until that table exists in the project pipeline, Q10 stays a **placeholder** or uses an agreed proxy. |

[↑ Back to summary](#summary)


In [ ]:

# ── Importing libraries ──────────────────────────────────────────────────────

import os

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter

from IPython.display import display

sns.set_theme(style="whitegrid")


def format_brl_value(value):
    formatted = f"{value:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return f"R$ {formatted}"


def format_brl(value, pos):
    return format_brl_value(value)



<a id="part-1"></a>

## Part 1 — RFM design & data loading

**Goal:** pull raw tables from Supabase, build **`df_tx`** (completed transactions + customer attributes), apply the **incomplete latest month** rule from notebook 2, and set **`as_of_date`**.

[↑ Back to summary](#summary)



<a id="q1"></a>

### 1. Load `customers_raw`, `transactions_raw`, `products_raw`, `customer_products_raw` from Supabase

[↑ Back to summary](#summary)


In [ ]:

# override=True ensures .env changes are picked up without restarting the kernel
load_dotenv(override=True)

DATABASE_URL = os.environ["SUPABASE_DATABASE_URL"]

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,  # helps avoid stale connections in notebooks
)

sql_customers = text(
    "SELECT\n"
    "  customer_id,\n"
    "  acquisition_channel,\n"
    "  acquisition_cost,\n"
    "  registration_date,\n"
    "  true_segment\n"
    "FROM public.customers_raw\n"
)

df_customers = pd.read_sql(sql_customers, engine)

sql_transactions = text(
    "SELECT\n"
    "  transaction_id,\n"
    "  customer_id,\n"
    "  transaction_datetime,\n"
    "  amount,\n"
    "  transaction_type,\n"
    "  product_type,\n"
    "  channel,\n"
    "  status\n"
    "FROM public.transactions_raw\n"
    "WHERE status = 'completed'\n"
)

df_transactions = pd.read_sql(sql_transactions, engine)

sql_products = text(
    "SELECT\n"
    "  product_id,\n"
    "  product_name,\n"
    "  product_type\n"
    "FROM public.products_raw\n"
)

df_products = pd.read_sql(sql_products, engine)

sql_customer_products = text(
    "SELECT\n"
    "  customer_id,\n"
    "  product_id,\n"
    "  start_date,\n"
    "  is_active\n"
    "FROM public.customer_products_raw\n"
)

df_customer_products = pd.read_sql(sql_customer_products, engine)

display(df_customers.head())
display(df_transactions.head())
display(df_products.head())
display(df_customer_products.head())
print("df_customers:", df_customers.shape)
print("df_transactions (completed):", df_transactions.shape)
print("df_products:", df_products.shape)
print("df_customer_products:", df_customer_products.shape)



<a id="q2"></a>

### 2. Join transactions to customer attributes

`df_transactions` is merged with `df_customers` on **`customer_id`** (inner join) so every row carries **`acquisition_channel`**, **`true_segment`**, and **`registration_date`**.

[↑ Back to summary](#summary)


In [ ]:
# Join transactions to customer attributes
df_tx = df_transactions.merge(df_customers, on="customer_id", how="inner")



<a id="q3"></a>

### 3. Transaction month buckets & incomplete latest month

We normalize **`transaction_datetime`** and **`registration_date`** to month-start timestamps, ensure **`amount`** is numeric, then **drop the trailing calendar month** when intra-month coverage is below the same threshold as [`2.EDA_cohort_analysis.ipynb`](2.EDA_cohort_analysis.ipynb) (**`min(20, days_in_month)`** distinct days with activity).

After that, **`as_of_date`** is the maximum **`transaction_datetime`** on retained rows.

[↑ Back to summary](#summary)


In [ ]:

# Transaction month (calendar month start)
tx_dt = df_tx["transaction_datetime"]
if isinstance(tx_dt.dtype, pd.DatetimeTZDtype):
    tx_dt = tx_dt.dt.tz_convert("UTC").dt.tz_localize(None)

df_tx["transaction_month"] = tx_dt.dt.to_period("M").dt.to_timestamp()

# Registration month (cohort month anchor)
reg = df_tx["registration_date"]
if isinstance(reg.dtype, pd.DatetimeTZDtype):
    reg = reg.dt.tz_convert("UTC").dt.tz_localize(None)

df_tx["registration_month"] = reg.dt.to_period("M").dt.to_timestamp()

# Amount numeric
df_tx["amount"] = pd.to_numeric(df_tx["amount"], errors="coerce")

# Drop the trailing calendar month if we don't yet have enough intra-month coverage.
# Rule: keep the latest month only if `transaction_datetime` includes activity on
# at least 20 distinct calendar days within that month (days 1–20+).
MIN_DISTINCT_DAYS_IN_LATEST_MONTH = 20

_tx_dt_guard = df_tx["transaction_datetime"]
if isinstance(_tx_dt_guard.dtype, pd.DatetimeTZDtype):
    _tx_dt_guard = _tx_dt_guard.dt.tz_convert("UTC").dt.tz_localize(None)

last_month_period = _tx_dt_guard.dt.to_period("M").max()
last_month_mask = _tx_dt_guard.dt.to_period("M") == last_month_period

distinct_days = _tx_dt_guard.loc[last_month_mask].dt.day.nunique()

days_in_month = last_month_period.days_in_month if pd.notna(last_month_period) else None
threshold = min(MIN_DISTINCT_DAYS_IN_LATEST_MONTH, days_in_month) if days_in_month is not None else MIN_DISTINCT_DAYS_IN_LATEST_MONTH

if pd.notna(last_month_period) and distinct_days < threshold:
    print(
        f"Dropping latest month {last_month_period} from df_tx: "
        f"only {distinct_days} distinct day(s) observed (threshold={threshold} of {days_in_month} days in month)."
    )
    df_tx = df_tx.loc[~last_month_mask].copy()
else:
    print(
        f"Keeping latest month {last_month_period}: "
        f"{distinct_days} distinct day(s) observed (threshold={threshold} of {days_in_month} days in month)."
    )

# Keep derived month columns consistent after potential row drops
_tx_dt2 = df_tx["transaction_datetime"]
if isinstance(_tx_dt2.dtype, pd.DatetimeTZDtype):
    _tx_dt2 = _tx_dt2.dt.tz_convert("UTC").dt.tz_localize(None)

df_tx["transaction_month"] = _tx_dt2.dt.to_period("M").dt.to_timestamp()

_reg2 = df_tx["registration_date"]
if isinstance(_reg2.dtype, pd.DatetimeTZDtype):
    _reg2 = _reg2.dt.tz_convert("UTC").dt.tz_localize(None)

df_tx["registration_month"] = _reg2.dt.to_period("M").dt.to_timestamp()

print("df_tx (joined):", df_tx.shape)
display(df_tx.head())

# Latest complete calendar month observed in completed transactions
latest_complete_month = df_tx["transaction_month"].max().to_period("M")
print("latest_complete_month:", latest_complete_month)

# Reference instant for RFM (12-month window ends here)
as_of_date = df_tx["transaction_datetime"].max()
print("as_of_date:", as_of_date)
